# 3B - Modeling su Dataset MACRO-BOND

## 3.1 - Import e Caricamento dati

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import f1_score, confusion_matrix, roc_auc_score, classification_report
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Caricamento del dataset
df_bond_macro_lagged = pd.read_csv('./data/df_bond_macro_lagged.csv', index_col=0 )

print(f"✓ Dataset preprocessato caricato: {df_bond_macro_lagged.shape[0]} righe e {df_bond_macro_lagged.shape[1]} colonne")
print(f"✓ Colonne disponibili: {df_bond_macro_lagged.columns.tolist()}")
print(f"NaN values per colonna:\n{df_bond_macro_lagged.isna().sum()}")

## 3.2 - Target Definition

A differenza dell'altro notebook questa volta proviamo a prevedere il prezzo del singolo bond e non della macroeconomia globale.


In [ ]:
FORECAST_HORIZON = 90

# Ordiniamo per ISIN e Data per garantire la corretta sequenza temporale
df_bond_macro_lagged = df_bond_macro_lagged.sort_values(by=['isincode', 'referencedate']).copy()

# Shiftiamo il prezzo per ogni ISIN in modo da creare il target di previsione a 90 giorni
df_bond_macro_lagged['pricevalue_target'] = df_bond_macro_lagged.groupby('isincode')['pricevalue'].shift(-FORECAST_HORIZON)

# Scartiamo le righe con valori NaN nel target
df_bond_macro_lagged = df_bond_macro_lagged.dropna(subset=['pricevalue_target']).copy()

# Creiamo la variabile target binaria: 1 se il prezzo aumenta, 0 altrimenti
df_bond_macro_lagged['target'] = (df_bond_macro_lagged['pricevalue_target'] > df_bond_macro_lagged['pricevalue']).astype(int)

print(f"\nTarget distribution:")
print(df_bond_macro_lagged['target'].value_counts())
print(f"Classe imbalance ratio: {(df_bond_macro_lagged['target'] == 1).sum() / (df_bond_macro_lagged['target'] == 0).sum():.3f}")

## 3.3 - Modello Naive: Persistenza statica 

In [ ]:
# Isoliamo X and y
y_true = df_bond_macro_lagged['target']

y_naive_static = np.zeros_like(y_true)  # Prevediamo sempre la classe 0 (Euribor_3M non aumenterà)
print("--- Baseline 1: Static Persistence ---")
print(f"\nClassification Report:\n{classification_report(y_true, y_naive_static, zero_division=0)}") # zero_division=0 per evitare warning in caso di classi non predette

Questi dati sono in linea con la distribuzione della classe di maggioranza. I modelli Più avanzati dovranno battere il **45% di accuratezza**.

## 3.4 - Modello Naive: Momentum Persistance

Valutiamo ora la persistenza del trend: predice che Euribor_3M aumenterà se è già in aumento negli ultimi 30 giorni. 

In [ ]:
# # Assumiamo di usare la stessa classe del lag a 63 giorni come previsione (90 giorni di calendario = 63 giorni lavorativi)
# # Scommetto 1 se il trend passato era in salita, 0 altrimenti

trend_lag_30 = df_bond_macro_lagged['pricevalue'] > df_bond_macro_lagged['pricevalue_lag_30d']
y_naive_trend = trend_lag_30.astype(int)

print("--- Baseline 2: Lagged Trend ---")
print(f"\nClassification Report:\n{classification_report(y_true, y_naive_trend, zero_division=0)}")
print(f"ROC-AUC: {roc_auc_score(y_true, y_naive_trend):.3f}")

Abbiamo un'accuratezza del **48%**

## 3.5 - Modello Random Forest

In [ ]:
# elenco le colonne presenti nel dataset
df_bond_macro_lagged.columns

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit

# Ordiniamo i dati per  Data per garantire la corretta sequenza temporale
df_bond_macro_lagged = df_bond_macro_lagged.sort_values(by=['referencedate']).reset_index(drop=True)

# Escludiamo dal dataset le colonne inutili per la previsione
columns_to_exclude = ['marketcode', 'referencedate', 'pricetype', 'pricevalue_target', 'target', 'redemptiondate', 'description']

feature_columns = [col for col in df_bond_macro_lagged.columns if col not in columns_to_exclude]

# Creiamo X e y
unique_dates = np.sort(df_bond_macro_lagged['referencedate'].unique())
n_splits = 5
test_size_days = len(unique_dates) // (n_splits + 1)
gap_days = FORECAST_HORIZON

roc_auc_scores = []
f1_scores = []

# Inizializziamo il modello Random Forest
rf_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)

for fold in range(n_splits):
    test_start_idx = len(unique_dates) - test_size_days * (n_splits - fold)
    test_end_idx = test_start_idx + test_size_days
    train_end_idx = test_start_idx - gap_days
    train_dates = unique_dates[:train_end_idx]
    test_dates = unique_dates[test_start_idx:test_end_idx]
    train_mask = df_bond_macro_lagged['referencedate'].isin(train_dates)
    test_mask = df_bond_macro_lagged['referencedate'].isin(test_dates)
    # 1. SPLIT: Dividiamo X e y usando le maschere temporali
    X_train = df_bond_macro_lagged.loc[train_mask, feature_columns].to_numpy(dtype=float)
    y_train = df_bond_macro_lagged.loc[train_mask, 'target'].to_numpy(dtype=int)
    X_test = df_bond_macro_lagged.loc[test_mask, feature_columns].to_numpy(dtype=float)
    y_test = df_bond_macro_lagged.loc[test_mask, 'target'].to_numpy(dtype=int)

    # 2. SCALING: Applichiamo lo standard scaler (fit solo sui dati di train)
    scaler = StandardScaler()
    
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # 3. TRAINING: Addestriamo il modello Random Forest sui dati di train scalati
    rf_model.fit(X_train_scaled, y_train)
    
    # 4. PREVISIONE 
    # predict() restiuisce 0 o 1 per l'F1-score
    y_pred = rf_model.predict(X_test_scaled)
    # predict_proba() restituisce le probabilità per ogni classe, usiamo la colonna della classe positiva (1) per il ROC-AUC
    y_proba = rf_model.predict_proba(X_test_scaled)[:, 1]

    # 5. VALUTAZIONE: Calcoliamo F1-score e ROC-AUC per questo fold
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_proba)
    f1_scores.append(f1)
    roc_auc_scores.append(roc_auc)

    print(f"Fold {fold+1}: Train size={len(X_train)}, Test size={len(X_test)}, F1-Score={f1:.3f}, ROC-AUC={roc_auc:.3f}")

print(f"\nRisultato Finale Random Forest (Media su {n_splits} folds):")
print(f"Mean ROC-AUC: {np.mean(roc_auc_scores):.3f}")
print(f"Mean F1-Score: {np.mean(f1_scores):.3f}")

## 3.6 - Modello LSTM (Long Short-Term Memory)

Creiamo la struttura della rete neurale

In [ ]:
# Creiamo la classe LSTM
class MacroLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout_rate):
        super(MacroLSTM, self).__init__()
        
        # layers LSTM
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout_rate) #  batch_first=True è FONDAMENTALE perché i nostri dati avranno forma (batch_size, seq_length, features)
        
        self.dropout = nn.Dropout(dropout_rate) # dropout per regolarizzazione

        # layer finale di classificazione
        self.fc = nn.Linear(hidden_size, 1)  # output binario
        self.sigmoid = nn.Sigmoid()  # per convertire l'output in probabilità

    def forward(self, x):
        # x shape: (batch, seq_len, features)
        lstm_out, (hn, cn) = self.lstm(x)  # lstm_out shape: (batch, seq_len, hidden_size)

        # Prendiamo solo l'output dell'ultimo timestep per la classificazione
        last_time_step_out = lstm_out[:, -1, :]  # shape: (batch, hidden_size)

        out = self.dropout(last_time_step_out)  # applichiamo dropout
        out = self.fc(out)  # shape: (batch, 1)
        return self.sigmoid(out)  # shape: (batch, 1) con valori tra 0 e 1

In [ ]:
def create_sequences(X_data, y_data, seq_length):
    """
    Trasforma array 2D in array 3D [samples, seq_length, features].
    """
    xs, ys = [], []
    for i in range(len(X_data) - seq_length + 1): # Aggiunto +1 per non perdere l'ultima riga
        # Prende la finestra di giorni (es. da 0 a 59)
        xs.append(X_data[i : (i + seq_length)])
        
        # Il target DEVE essere quello associato all'ULTIMO giorno della finestra
        # L'ultimo giorno della finestra è (i + seq_length - 1)
        ys.append(y_data[i + seq_length - 1]) 
        
    return np.array(xs), np.array(ys)

In [ ]:
# Importiamo il dataset macro senza lag
df_bond_macro_pp = pd.read_csv('./data/df_bond_macro.csv', index_col=0).reset_index()

# Assicuriamoci che i nomi siano corretti (se reset_index la chiama 'index', la rinominiamo)
if 'index' in df_bond_macro_pp.columns:
    df_bond_macro_pp = df_bond_macro_pp.rename(columns={'index': 'isincode'})
    print("✓ Colonna 'index' rinominata in 'isincode'")

# Definiamo la lunghezza dell'orizzonte di previsione
SEQ_LENGTH = 90

# Assicuriamoci che le date siano datetime e ordiniamo temporalmente l'intero dataset
df_bond_macro_pp['referencedate'] = pd.to_datetime(df_bond_macro_pp['referencedate'])
df_bond_macro_pp = df_bond_macro_pp.sort_values(['referencedate', 'isincode'])

# Creiamo la colonna col valore futuro
df_bond_macro_pp['pricevalue_target'] = df_bond_macro_pp.groupby('isincode')['pricevalue'].shift(-FORECAST_HORIZON)
df_bond_macro_pp['target'] = df_bond_macro_pp['pricevalue_target'] > df_bond_macro_pp['pricevalue']

# Droppiamo i valori NaN creati dallo shift
df_bond_macro_pp = df_bond_macro_pp.dropna(subset=['pricevalue_target']).copy()

# Selezioniamo le feature base
feature_cols = ['pricevalue', 'volume',
       'mintoday', 'maxtoday', 'coupon',
       'days_to_maturity', 'years_to_maturity', 'Yield_to_Maturity',
       'ecb_deposit_rate', 'ecb_mro_rate', 'fed_rate', 'vix', 'esi_index',
       'euribor_3m', 'euribor_1y', 'hicp_euroarea', 'stoxx50', 'xeon', 'sega',
       'interbank_stress_spread', 'short_yield_curve_slope', 'bce_fed_spread']

unique_dates = np.sort(df_bond_macro_pp['referencedate'].unique())
n_splits = 5
test_size_days = len(unique_dates) // (n_splits + 2)
gap_days = SEQ_LENGTH

lstm_roc_auc_scores = []
lstm_f1_scores = []

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

for fold in range(n_splits):
       print(f"\n--- Fold {fold+1} ---")
    
       test_start_idx = (len(unique_dates) - test_size_days * (n_splits - fold)) - (SEQ_LENGTH - 1) # i dati precedenti devono essere inclusi
       test_end_idx = test_start_idx + test_size_days
       train_end_idx = test_start_idx - gap_days
       train_dates = unique_dates[:train_end_idx]
       test_dates = unique_dates[test_start_idx:test_end_idx]
       train_mask = df_bond_macro_pp['referencedate'].isin(train_dates)
       test_mask = df_bond_macro_pp['referencedate'].isin(test_dates)
    
       # SPLIT: Dividiamo il DataFrame mantenendo tutte le colonne
       df_train = df_bond_macro_pp[train_mask].copy()
       df_test = df_bond_macro_pp[test_mask].copy()

       # SCALING: Facciamo fit solo sul train
       scaler = StandardScaler()
       df_train[feature_cols] = scaler.fit_transform(df_train[feature_cols])
       df_test[feature_cols] = scaler.transform(df_test[feature_cols])

       # CREAZIONE SEQUENZE: Per ogni ISIN creiamo le sequenze di input per LSTM
       X_train_seq_list, y_train_seq_list = [], []
       X_test_seq_list, y_test_seq_list = [], []

       # Ciclo per train
       for isin in df_train['isincode'].unique():
              df_isin = df_train[df_train['isincode'] == isin].sort_values(by='referencedate')

              # creiamo sequenze solo se ci sono abbastanza giorni
              if len(df_isin) > SEQ_LENGTH:
                     X_seq, y_seq = create_sequences(df_isin[feature_cols].values, df_isin['target'].values, SEQ_LENGTH)
                     X_train_seq_list.append(X_seq)
                     y_train_seq_list.append(y_seq)

       # Ciclo per Test
       for isin in df_test['isincode'].unique():
              df_isin = df_test[df_test['isincode'] == isin].sort_values(by='referencedate')

              # creiamo sequenze solo se ci sono abbastanza giorni
              if len(df_isin) > SEQ_LENGTH:
                     X_seq, y_seq = create_sequences(df_isin[feature_cols].values, df_isin['target'].values, SEQ_LENGTH)
                     X_test_seq_list.append(X_seq)
                     y_test_seq_list.append(y_seq)

        # Ciclo per Test
       for isin in df_test['isincode'].unique():
              df_isin = df_test[df_test['isincode'] == isin].sort_values(by='referencedate')
              if len(df_isin) > SEQ_LENGTH:
                     X_seq, y_seq = create_sequences(df_isin[feature_cols].values, df_isin['target'].values, SEQ_LENGTH)
                     X_test_seq_list.append(X_seq)
                     y_test_seq_list.append(y_seq)

       # --- SAFETY CHECK ---
       if not X_train_seq_list or not X_test_seq_list:
           print("ERRORE: Non ci sono abbastanza giorni per creare le sequenze in questo Fold. Salto...")
           continue
       
       # Concateniamo tutte le sequenze
       X_train_3d = np.concatenate(X_train_seq_list)
       y_train_seq = np.concatenate(y_train_seq_list)
       X_test_3d = np.concatenate(X_test_seq_list)
       y_test_seq = np.concatenate(y_test_seq_list)

       # Convertiamo in tensori PyTorch
       X_train_tensor = torch.tensor(X_train_3d, dtype=torch.float32)
       y_train_tensor = torch.tensor(y_train_seq, dtype=torch.float32).unsqueeze(1) # Aggiungiamo dimensione per BCE Loss
       X_test_tensor = torch.tensor(X_test_3d, dtype=torch.float32)
       y_test_tensor = torch.tensor(y_test_seq, dtype=torch.float32).unsqueeze(1) # Aggiungiamo dimensione per BCE Loss

       # DataLoaders (per addestrare a "pacchetti" e non saturare la RAM)
       train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=64, shuffle=True)
       test_loader = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=64, shuffle=False)

       # Inizializziamo il modello per QUESTO fold
       input_dim = X_train_3d.shape[2]
       torch.set_grad_enabled(True)
       model = MacroLSTM(input_size=input_dim, hidden_size=32, num_layers=1, dropout_rate=0.2)
       model.to(device)  # Spostiamo il modello sul device disponibile (CPU o GPU)

       # Loss per classificazione binaria e ottimizzatore Adam
       criterion = nn.BCELoss()
       optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-4)
       
       epochs = 23  

       # --- TRAINING ---
       for epoch in range(epochs):
              model.train()
              train_loss = 0.0
              for batch_X, batch_y in train_loader:
                     batch_X, batch_y = batch_X.to(device), batch_y.to(device)  # Spostiamo i batch sul device
                     optimizer.zero_grad()
                     outputs = model(batch_X)
                     loss = criterion(outputs, batch_y)
                     loss.backward()
                     optimizer.step()
                     train_loss += loss.item()     

       # --- EVALUATION SUL TEST SET DEL FOLD ---
       model.eval()
       y_preds_prob = []
       y_trues = []
    
       with torch.no_grad():
              for batch_X, batch_y in test_loader:
                     batch_X, batch_y = batch_X.to(device), batch_y.to(device)  # Spostiamo i batch sul device
                     probs = model(batch_X)
                     
                     # Usiamo .cpu() per riportare i dati nella memoria normale prima di passarli a scikit-learn!
                     y_preds_prob.extend(probs.cpu().numpy().flatten())
                     y_trues.extend(batch_y.cpu().numpy().flatten())
            
       # Calcolo ROC-AUC (Indipendente dalla soglia, misura il vero potere predittivo)
       roc = roc_auc_score(y_trues, y_preds_prob)
       
       # Converti probabilità in 0 o 1 usando la soglia dinamica
       y_preds_class = [1 if p > 0.5 else 0 for p in y_preds_prob]
       
       # Calcola F1-Score
       f1 = f1_score(y_trues, y_preds_class)
       
       print(f"Fold ROC-AUC: {roc:.3f} | F1: {f1:.3f}")
       print(f"Confusion Matrix Fold {fold+1}:\n", confusion_matrix(y_trues, y_preds_class))
       
       # Salviamo le metriche per il fold corrente
       lstm_roc_auc_scores.append(roc)
       lstm_f1_scores.append(f1)

# STAMPA FINALE
print(f"\n=======================================================")
print(f"Risultato Finale LSTM (Media su {n_splits} folds):")
print(f"Mean ROC-AUC: {np.mean(lstm_roc_auc_scores):.3f}")
print(f"Mean F1-Score: {np.mean(lstm_f1_scores):.3f}")
print(f"=======================================================")